In [1]:
import warnings

warnings.filterwarnings('ignore')

In [2]:
import polars as pl
from math import log
import matplotlib.pyplot as plt
import os
from sklearn.preprocessing import MinMaxScaler

In [3]:
trips_df = pl.read_csv(source="data/trips_dataset.csv", infer_schema_length=10000)

individuals_df = pl.read_csv(source="data/individuals_dataset.csv", infer_schema_length=10000)

In [4]:
trips_df.shape, individuals_df.shape

((80697, 27), (3337, 27))

In [5]:
individuals_with_trips_df = individuals_df
#.join(other=trips_df, on=["ID"], how="inner", suffix="_right", coalesce=True))

In [6]:
individuals_with_trips_df

ID,CODGEO,AREA_NAME,SEX,AGE,DIPLOMA,PRO_CAT,TYPE_HOUSE,NBPERS_HOUSE,NB_10,NB_11_17,NB_18_24,NB_25_64,NB_65,PMR,DRIVING_LICENCE,NB_CAR,TWO_WHEELER,BIKE,ELECT_SCOOTER,NAVIGO_SUB,IMAGINER_SUB,OTHER_SUB_PT,BIKE_SUB,NSM_SUB,WEIGHT_INDIV,GPS_RECORD
str,i64,str,str,i64,str,i64,str,i64,i64,f64,f64,i64,i64,bool,bool,i64,str,str,str,bool,bool,bool,bool,bool,f64,bool
"""10_2978""",78092,"""Bougival""","""Woman""",41,null,4,"""In a couple with child(ren)""",2,1,0.0,0.0,1,0,false,true,2,"""1""","""2""","""0""",false,false,false,false,false,1856.20616,true
"""10_2980""",75120,"""Paris""","""Man""",30,"""5-year-and-above higher educat…",2,"""Living alone""",1,0,0.0,0.0,0,0,false,true,0,"""0""","""0""","""0""",true,false,false,false,false,1375.000372,true
"""10_2981""",91326,"""Juvisy-sur-Orge""","""Man""",38,"""5-year-and-above higher educat…",2,"""In a couple without children""",2,0,0.0,0.0,2,0,false,true,1,"""0""","""1""","""0""",true,false,false,false,false,1231.81299,true
"""10_2982""",91573,"""Saint-Pierre-du-Perray""","""Man""",43,"""3–4-year higher education degr…",2,"""Single parent (divorced / sepa…",1,1,0.0,0.0,0,0,false,true,1,"""0""","""2""","""0""",true,false,false,false,false,426.311616,true
"""10_2984""",78073,"""Bois-d'Arcy""","""Woman""",39,"""5-year-and-above higher educat…",2,"""Living alone""",1,0,0.0,0.0,0,0,false,true,1,"""0""","""1""","""0""",true,false,false,true,false,843.194726,true
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""7_2972""",95127,"""Cergy""","""Woman""",59,"""5-year-and-above higher educat…",2,"""In a couple without children""",1,0,0.0,0.0,0,1,true,true,1,"""0""","""0""","""0""",false,false,false,false,false,608.890788,true
"""7_2973""",91461,"""Ollainville""","""Woman""",42,null,2,"""In a couple with child(ren)""",5,1,2.0,0.0,2,0,true,true,2,"""0""","""2""","""0""",false,false,false,false,false,786.094977,true
"""7_2974""",91534,"""Saclay""","""Man""",52,"""3–4-year higher education degr…",2,"""In a couple with child(ren)""",3,0,0.0,1.0,2,0,false,true,1,"""0""","""2""","""0""",false,false,false,false,false,1819.688178,true


In [7]:
individuals_with_trips_df.describe()

statistic,ID,CODGEO,AREA_NAME,SEX,AGE,DIPLOMA,PRO_CAT,TYPE_HOUSE,NBPERS_HOUSE,NB_10,NB_11_17,NB_18_24,NB_25_64,NB_65,PMR,DRIVING_LICENCE,NB_CAR,TWO_WHEELER,BIKE,ELECT_SCOOTER,NAVIGO_SUB,IMAGINER_SUB,OTHER_SUB_PT,BIKE_SUB,NSM_SUB,WEIGHT_INDIV,GPS_RECORD
str,str,f64,str,str,f64,str,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,f64,f64,f64,f64,f64,f64,f64
"""count""","""3337""",3337.0,"""3337""","""3337""",3337.0,"""2811""",3337.0,"""3337""",3337.0,3337.0,3337.0,3337.0,3337.0,3337.0,3337.0,3322.0,3337.0,"""3337""","""3337""","""3337""",3248.0,3337.0,3337.0,3337.0,3337.0,3337.0,3337.0
"""null_count""","""0""",0.0,"""0""","""0""",0.0,"""526""",0.0,"""0""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,15.0,0.0,"""0""","""0""","""0""",89.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",null,85883.256218,null,null,42.495055,null,3.996404,null,2.48217,0.39976,0.320947,0.293078,1.213665,0.119568,0.034162,0.841361,1.033563,null,null,null,0.534483,0.103386,0.053042,0.10938,0.019479,2710.244737,0.994906
"""std""",null,8272.53916,null,null,14.857539,null,2.064891,null,1.48788,0.823937,0.649955,0.612463,0.880895,0.410917,null,null,0.897679,null,null,null,null,null,null,null,null,2826.691074,null
"""min""","""10_2978""",75101.0,"""Ablis""","""Man""",16.0,"""3–4-year higher education degr…",1.0,"""Another family member in the h…",1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""0""","""0""","""0""",0.0,0.0,0.0,0.0,0.0,343.084713,0.0
"""25%""",null,77217.0,null,null,30.0,null,2.0,null,1.0,0.0,0.0,0.0,0.0,0.0,null,null,0.0,null,null,null,null,null,null,null,null,962.52523,null
"""50%""",null,91363.0,null,null,42.0,null,3.0,null,2.0,0.0,0.0,0.0,1.0,0.0,null,null,1.0,null,null,null,null,null,null,null,null,1713.254362,null
"""75%""",null,93051.0,null,null,53.0,null,6.0,null,4.0,0.0,0.0,0.0,2.0,0.0,null,null,2.0,null,null,null,null,null,null,null,null,3133.977803,null
"""max""","""7_2976""",95680.0,"""Yerres""","""Woman""",82.0,"""Vocational certificate (CAP, B…",8.0,"""Single parent (divorced / sepa…",12.0,4.0,4.0,4.0,4.0,4.0,1.0,1.0,8.0,"""4+""","""4+""","""4+""",1.0,1.0,1.0,1.0,1.0,19939.046556,1.0


In [8]:
individuals_with_trips_df.null_count()

ID,CODGEO,AREA_NAME,SEX,AGE,DIPLOMA,PRO_CAT,TYPE_HOUSE,NBPERS_HOUSE,NB_10,NB_11_17,NB_18_24,NB_25_64,NB_65,PMR,DRIVING_LICENCE,NB_CAR,TWO_WHEELER,BIKE,ELECT_SCOOTER,NAVIGO_SUB,IMAGINER_SUB,OTHER_SUB_PT,BIKE_SUB,NSM_SUB,WEIGHT_INDIV,GPS_RECORD
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,526,0,0,0,0,0,0,0,0,0,15,0,0,0,0,89,0,0,0,0,0,0


In [9]:
dtypes_dict = {col: individuals_with_trips_df[col].dtype for col in individuals_with_trips_df.columns}
dtypes_dict


{'ID': String,
 'CODGEO': Int64,
 'AREA_NAME': String,
 'SEX': String,
 'AGE': Int64,
 'DIPLOMA': String,
 'PRO_CAT': Int64,
 'TYPE_HOUSE': String,
 'NBPERS_HOUSE': Int64,
 'NB_10': Int64,
 'NB_11_17': Float64,
 'NB_18_24': Float64,
 'NB_25_64': Int64,
 'NB_65': Int64,
 'PMR': Boolean,
 'DRIVING_LICENCE': Boolean,
 'NB_CAR': Int64,
 'TWO_WHEELER': String,
 'BIKE': String,
 'ELECT_SCOOTER': String,
 'NAVIGO_SUB': Boolean,
 'IMAGINER_SUB': Boolean,
 'OTHER_SUB_PT': Boolean,
 'BIKE_SUB': Boolean,
 'NSM_SUB': Boolean,
 'WEIGHT_INDIV': Float64,
 'GPS_RECORD': Boolean}

In [10]:
individuals_with_trips_df.filter(pl.col("GPS_RECORD") == False).height / individuals_with_trips_df.shape[0] / 100


5.0943961642193586e-05

In [11]:
individuals_with_trips_df = individuals_with_trips_df.filter(pl.col("GPS_RECORD") == True)

# individuals_with_trips_df = individuals_with_trips_df.with_columns([
#     pl.col("Date_EMG").str.to_date(strict=False),
#     pl.col("Date_O").str.to_date(strict=False),
#     pl.col("Date_D").str.to_date(strict=False),
#     pl.col("Time_O").str.to_time(strict=False),
#     pl.col("Time_D").str.to_time(strict=False),
# ]
# )

## Encoding Values & Preparing Values for building social vulnerability index (SVI)

### Converting Values to be ready for calculations

In [12]:
"""

for col in are_correlated.select(cs.string()):
        print(col.name ,"  ---------->  " , col.n_unique())

#OUTPUT:
    SEX   ---------->   2
    DIPLOMA   ---------->   7
    TWO_WHEELER   ---------->   5
    BIKE   ---------->   5
    ELECT_SCOOTER   ---------->   5

"""
individuals_with_trips_df = individuals_with_trips_df.with_columns(
    [
        pl.col("SEX").replace({"Man": 0, "Woman": 1}, return_dtype=pl.Int8),

        pl.col("DIPLOMA").replace({
            "5-year-and-above higher education degree: Master's 2, DEA, DESS, Grande École Diploma, Doctorate, etc.": 6,
            "3–4-year higher education degree: Licence, Professional Licence, Master 1, or equivalent": 5,
            "Upper secondary diploma (Baccalauréat) or equivalent": 4,
            "Vocational certificate (CAP, BEP) or equivalent": 3,
            "Lower secondary certificate (Brevet) or equivalent": 2,
            "No diploma": 1

        }, default=0).cast(pl.Int8),

        pl.col("TWO_WHEELER").replace({"4+": 4}, return_dtype=pl.Int8),
        pl.col("BIKE").replace({"4+": 4}, return_dtype=pl.Int8),
        pl.col("ELECT_SCOOTER").replace({"4+": 4}, return_dtype=pl.Int8),

        #"NAVIGO_SUB", "IMAGINER_SUB", "OTHER_SUB_PT" , "BIKE_SUB", "NSM_SUB"
        pl.col("NAVIGO_SUB").replace({0: 1, 1: 0}, default=0).cast(pl.Int8),
        pl.col("IMAGINER_SUB").replace({0: 1, 1: 0}),
        pl.col("OTHER_SUB_PT").replace({0: 1, 1: 0}),
        pl.col("BIKE_SUB").replace({0: 1, 1: 0}),
        pl.col("NSM_SUB").replace({0: 1, 1: 0}),

    ]
)

### Applying the decreasing function 1/1+log(x) according to the described relations bet. the features applied on it & the Social Vulnerability Index

In [13]:
mean_age = individuals_with_trips_df["AGE"].mean()
individuals_with_trips_df = individuals_with_trips_df.with_columns(
    [
        pl.col("AGE").map_elements(lambda
                                       x: log(1 + abs(x - mean_age))

                                   ).alias('TRANSFORMED_AGE'),
    ]
)

individuals_with_trips_df = individuals_with_trips_df.with_columns([
    pl.col("NB_CAR").map_elements(lambda x: 1 / (1 + log(1 + x))).alias('TRANSFORMED_NB_CAR'),
    #added the 1 in the log to avoid the people with no cars
    # pl.col("PRO_CAT").map_elements(lambda x: 1 / (1 + log(x))).alias('TRANSFORMED_PRO_CAT'),
    pl.col("NBPERS_HOUSE").map_elements(lambda x: (log(x))).alias('TRANSFORMED_NBPERS_HOUSE'),
    pl.col("TWO_WHEELER").map_elements(lambda x: 1 / (1 + log(1 + x))).alias('TRANSFORMED_TWO_WHEELER'),
    pl.col("BIKE").map_elements(lambda x: 1 / (1 + log(1 + x))).alias('TRANSFORMED_BIKE'),
    pl.col("ELECT_SCOOTER").map_elements(lambda x: 1 / (1 + log(1 + x))).alias('TRANSFORMED_ELECT_SCOOTER'),
    pl.col("DIPLOMA").map_elements(lambda x: 1 / (1 + log(1 + x))).alias('TRANSFORMED_DIPLOMA'),

])

individuals_with_trips_df = individuals_with_trips_df.drop("PRO_CAT", strict=False)

### Converting Bool Values to Integers

In [14]:
individuals_with_trips_df.cast(dtypes={pl.Boolean: pl.Int8})

ID,CODGEO,AREA_NAME,SEX,AGE,DIPLOMA,TYPE_HOUSE,NBPERS_HOUSE,NB_10,NB_11_17,NB_18_24,NB_25_64,NB_65,PMR,DRIVING_LICENCE,NB_CAR,TWO_WHEELER,BIKE,ELECT_SCOOTER,NAVIGO_SUB,IMAGINER_SUB,OTHER_SUB_PT,BIKE_SUB,NSM_SUB,WEIGHT_INDIV,GPS_RECORD,TRANSFORMED_AGE,TRANSFORMED_NB_CAR,TRANSFORMED_NBPERS_HOUSE,TRANSFORMED_TWO_WHEELER,TRANSFORMED_BIKE,TRANSFORMED_ELECT_SCOOTER,TRANSFORMED_DIPLOMA
str,i64,str,i8,i64,i8,str,i64,i64,f64,f64,i64,i64,i8,i8,i64,i8,i8,i8,i8,i8,i8,i8,i8,f64,i8,f64,f64,f64,f64,f64,f64,f64
"""10_2978""",78092,"""Bougival""",1,41,0,"""In a couple with child(ren)""",2,1,0.0,0.0,1,0,0,1,2,1,2,0,1,1,1,1,1,1856.20616,1,0.921458,0.476505,0.693147,0.590616,0.476505,1.0,1.0
"""10_2980""",75120,"""Paris""",0,30,6,"""Living alone""",1,0,0.0,0.0,0,0,0,1,0,0,0,0,0,1,1,1,1,1375.000372,1,2.603649,1.0,0.0,1.0,1.0,1.0,0.339454
"""10_2981""",91326,"""Juvisy-sur-Orge""",0,38,6,"""In a couple without children""",2,0,0.0,0.0,2,0,0,1,1,0,1,0,0,1,1,1,1,1231.81299,1,1.7071,0.590616,0.693147,1.0,0.590616,1.0,0.339454
"""10_2982""",91573,"""Saint-Pierre-du-Perray""",0,43,5,"""Single parent (divorced / sepa…",1,1,0.0,0.0,0,0,0,1,1,0,2,0,0,1,1,1,1,426.311616,1,0.396793,0.590616,0.0,1.0,0.476505,1.0,0.358197
"""10_2984""",78073,"""Bois-d'Arcy""",1,39,6,"""Living alone""",1,0,0.0,0.0,0,0,0,1,1,0,1,0,0,1,1,0,1,843.194726,1,1.506951,0.590616,0.0,1.0,0.590616,1.0,0.339454
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""7_2972""",95127,"""Cergy""",1,59,6,"""In a couple without children""",1,0,0.0,0.0,0,1,1,1,1,0,0,0,1,1,1,1,1,608.890788,1,2.861461,0.590616,0.0,1.0,1.0,1.0,0.339454
"""7_2973""",91461,"""Ollainville""",1,42,0,"""In a couple with child(ren)""",5,1,2.0,0.0,2,0,1,1,2,0,2,0,1,1,1,1,1,786.094977,1,0.414063,0.476505,1.609438,1.0,0.476505,1.0,1.0
"""7_2974""",91534,"""Saclay""",0,52,5,"""In a couple with child(ren)""",3,0,0.0,1.0,2,0,0,1,1,0,2,0,1,1,1,1,1,1819.688178,1,2.350141,0.590616,1.098612,1.0,0.476505,1.0,0.358197


## Extensive Data Analysis

In [15]:
# ProfileReport(individuals_with_trips_df.to_pandas(),minimal=False , tsmode=False, title="Profiling Report" , explorative=True).to_file("report_after_encoding.html")


In [16]:
are_correlated = individuals_with_trips_df.select(
    pl.col(
        "SEX", "TRANSFORMED_AGE", "TRANSFORMED_DIPLOMA", "TRANSFORMED_NB_CAR"
        , "PMR", "TRANSFORMED_NBPERS_HOUSE",
        "TRANSFORMED_TWO_WHEELER", "TRANSFORMED_BIKE", "TRANSFORMED_ELECT_SCOOTER",
        "NAVIGO_SUB", "IMAGINER_SUB", "OTHER_SUB_PT", "BIKE_SUB", "NSM_SUB"
    )
)

are_correlated.describe()

are_correlated_raw = are_correlated.__deepcopy__()

## Making It Ready to Build SVI (Social Vulnerability Index)

In [17]:
cols_2_continue_operations_on = are_correlated.columns


### Normalizing All Features separately by min_max scaler to make each of them particicpate equally in the MVI

### Investigating which dimensionality reduction technique is the best

In [31]:
import polars as pl
from sklearn.decomposition import KernelPCA


def compare_dim_reduction_variances(df: pl.DataFrame, cols: list[str]):
    """
    Compare variance explained by different dimensionality reduction methods.

    Parameters:
    -----------
    df : polars DataFrame with vulnerability data
    cols : list of column names to use

    Returns:
    --------
    dict with variance explained by each method
    """
    # Get data and standardize
    X = df.select(cols).to_numpy()
    X_scaled = StandardScaler().fit_transform(X)

    # Total variance in standardized data
    total_variance = X_scaled.var(axis=0).sum()

    results = {}

    # PCA
    pca = PCA(n_components=1)
    pca.fit(X_scaled)
    pca_variance = pca.explained_variance_ratio_[0]

    results["PCA"] = {
        "variance_explained": pca_variance,
        "method": "PCA captures linear variance. Baseline method."
    }

    # Test multiple KernelPCA variants
    kernels = [
        ("linear", {}),
        ("rbf", {}),
        ("poly", {}),
        ("sigmoid", {}),
        ("cosine", {})
    ]

    for kernel_name, params in kernels:
        try:
            kpca = KernelPCA(n_components=1, kernel=kernel_name, **params)
            kpca_proj = kpca.fit_transform(X_scaled).flatten()

            # For KernelPCA, we calculate variance of projection as proxy
            kpca_variance = np.var(kpca_proj) / total_variance

            results[f"KernelPCA_{kernel_name}"] = {
                "variance_explained": kpca_variance,
                "method": f"Nonlinear {kernel_name} kernel"
            }
        except Exception as e:
            results[f"KernelPCA_{kernel_name}"] = {"error": str(e)}

    # Factor Analysis
    try:
        fa = FactorAnalysis(n_components=1, random_state=42)
        fa.fit(X_scaled)

        # Calculate variance explained by first factor
        loadings = fa.components_.T  # Shape: (n_features, n_factors)
        fa_variance = np.sum(loadings[:, 0] ** 2) / total_variance

        results["FactorAnalysis"] = {
            "variance_explained": fa_variance,
            "method": "Models shared variance among features"
        }
    except Exception as e:
        results["FactorAnalysis"] = {"error": str(e)}

    # ICA
    try:
        ica = FastICA(n_components=1, random_state=42)
        ica.fit(X_scaled)
        ica_proj = ica.transform(X_scaled).flatten()

        # For ICA, we calculate variance of projection
        ica_variance = np.var(ica_proj) / total_variance

        results["ICA"] = {
            "variance_explained": ica_variance,
            "method": "Independent components; not variance-maximizing"
        }
    except Exception as e:
        results["ICA"] = {"error": str(e)}

    return results


compare_dim_reduction_variances(are_correlated_raw, cols_2_continue_operations_on)

In [38]:
import polars as pl
from sklearn.decomposition import FastICA, FactorAnalysis


def extract_svi_feature_weights(df: pl.DataFrame, cols: list[str]):
    """
    Extract feature weights from different dimensionality reduction methods for SVI construction.
    Returns both raw weights and absolute values for importance ranking.

    Parameters:
    -----------
    df : polars DataFrame with vulnerability data
    cols : list of column names to use

    Returns:
    --------
    dict with weights from each method
    """
    # Get data and standardize
    X = df.select(cols).to_numpy()
    X_scaled = StandardScaler().fit_transform(X)

    results = {}

    # PCA
    pca = PCA(n_components=1)
    pca.fit(X_scaled)
    pca_weights = pca.components_[0]

    # Convert to pandas Series for easier handling
    pca_weights_series = pd.Series(pca_weights, index=cols)

    # Apply sign flip check if needed (assuming first feature should have positive weight)
    if len(cols) > 0 and pca_weights_series.iloc[0] < 0:
        pca_weights_series = pca_weights_series * -1

    results["PCA"] = {
        # "raw_weights": pca_weights_series.to_dict(),
        "abs_weights": pca_weights_series.abs().to_dict(),
        "method": "Principal Component Analysis - weights represent linear combination maximizing variance"
    }

    # ICA
    try:
        ica = FastICA(n_components=1, random_state=42)
        ica.fit(X_scaled)
        ica_weights = ica.components_[0]

        ica_weights_series = pd.Series(ica_weights, index=cols)

        # Apply sign flip check
        if len(cols) > 0 and ica_weights_series.iloc[0] < 0:
            ica_weights_series = ica_weights_series * -1

        results["ICA"] = {
            # "raw_weights": ica_weights_series.to_dict(),
            "abs_weights": ica_weights_series.abs().to_dict(),
            "method": "Independent Component Analysis - weights represent statistically independent sources"
        }
    except Exception as e:
        results["ICA"] = {"error": str(e)}

    # Factor Analysis
    try:
        fa = FactorAnalysis(n_components=1, random_state=42)
        fa.fit(X_scaled)
        fa_weights = fa.components_[0]

        fa_weights_series = pd.Series(fa_weights, index=cols)

        # Apply sign flip check
        if len(cols) > 0 and fa_weights_series.iloc[0] < 0:
            fa_weights_series = fa_weights_series * -1

        results["FactorAnalysis"] = {
            # "raw_weights": fa_weights_series.to_dict(),
            "abs_weights": fa_weights_series.abs().to_dict(),
            "method": "Factor Analysis - weights represent loadings on latent vulnerability factor"
        }
    except Exception as e:
        results["FactorAnalysis"] = {"error": str(e)}

    return results


extract_svi_feature_weights(are_correlated_raw, cols_2_continue_operations_on)

{'PCA': {'abs_weights': {'SEX': 0.027023826319088104,
   'TRANSFORMED_AGE': 0.28085521095777155,
   'TRANSFORMED_DIPLOMA': 0.12272036210978322,
   'TRANSFORMED_NB_CAR': 0.5001728729087579,
   'PMR': 0.06554781344347349,
   'TRANSFORMED_NBPERS_HOUSE': 0.4861448745362952,
   'TRANSFORMED_TWO_WHEELER': 0.2597430473348936,
   'TRANSFORMED_BIKE': 0.4113670956712334,
   'TRANSFORMED_ELECT_SCOOTER': 0.19555704794716677,
   'NAVIGO_SUB': 0.2991523414704307,
   'IMAGINER_SUB': 0.08954554632721616,
   'OTHER_SUB_PT': 0.029307975793498392,
   'BIKE_SUB': 0.20213687438523142,
   'NSM_SUB': 0.01904782366615386},
  'method': 'Principal Component Analysis - weights represent linear combination maximizing variance'},
 'ICA': {'abs_weights': {'SEX': 0.01907377822332105,
   'TRANSFORMED_AGE': 0.19823136603304384,
   'TRANSFORMED_DIPLOMA': 0.08661767370501067,
   'TRANSFORMED_NB_CAR': 0.35302870653976615,
   'PMR': 0.046264523827306094,
   'TRANSFORMED_NBPERS_HOUSE': 0.34312755757906277,
   'TRANSFORMED_

In [22]:
import polars as pl
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA


def assess_social_vulnerability_weights(df: pl.DataFrame, cols: list[str], min_variance_threshold=0.99):
    """
    Assess appropriate dimensionality reduction for social vulnerability index creation.
    Focuses on interpretable weights with proper variance capture.

    Parameters
    ----------
    df : pl.DataFrame
        Input data containing vulnerability indicators
    cols : list[str]
        Column names of vulnerability indicators
    min_variance_threshold : float, optional
        Minimum cumulative variance to capture (default: 0.7)

    Returns
    -------
    results : dict
        Dictionary with analysis results and recommended weights
    """
    # Extract numeric data
    X = df.select(cols).to_numpy()

    # Standardize data
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    results = {
        "feature_names": cols,
        "n_features": len(cols),
        "min_variance_threshold": min_variance_threshold
    }

    # -------------------- PCA Analysis --------------------
    # Determine optimal number of components
    pca = PCA()
    pca.fit(X_scaled)

    # Find how many components needed to reach threshold
    cumulative_var = np.cumsum(pca.explained_variance_ratio_)
    n_components = np.argmax(cumulative_var >= min_variance_threshold) + 1

    # Refit with optimal components
    pca_final = PCA(n_components=n_components)
    pca_final.fit(X_scaled)

    # Extract weights for the vulnerability index
    # For social vulnerability, often the first component is used
    # but we should verify if it makes domain sense
    component_weights = pca_final.components_[0]

    results["pca_analysis"] = {
        "n_components_needed": n_components,
        "cumulative_variance": cumulative_var[n_components - 1],
        "component_weights": dict(zip(cols, component_weights.tolist())),
        "explained_variance_ratio": pca_final.explained_variance_ratio_.tolist(),
        "loading_matrix": pca_final.components_.tolist(),
        "note": "Weights are directly interpretable. First component often used for SVI when it captures meaningful structure."
    }

    # -------------------- Weight Validation --------------------
    # Check if weights align with expected vulnerability directions
    # (This would be customized based on domain knowledge)
    results["weight_validation"] = {
        "has_expected_signs": "Requires domain verification",
        "suggested_review": "Verify that weights for known positive indicators (e.g., poverty) have positive weights and negative indicators (e.g., education) have negative weights"
    }

    # -------------------- Recommendation --------------------
    if n_components == 1:
        recommendation = "Single-component PCA is appropriate as it captures sufficient variance."
    else:
        recommendation = f"Consider using first component (captures {pca.explained_variance_ratio_[0]:.1%} variance) or creating a composite index with {n_components} components."

    results["recommendation"] = {
        "method": "PCA",
        "reason": "PCA provides directly interpretable weights essential for social vulnerability indices where transparency is critical.",
        "details": recommendation,
        "next_steps": [
            "Validate weight directions against domain knowledge",
            "Consider normalizing weights to sum to 1 for easier interpretation",
            "Test correlation between index and known vulnerability outcomes if available"
        ]
    }

    return results


assess_social_vulnerability_weights(are_correlated_raw, cols_2_continue_operations_on)

NameError: name 'np' is not defined

In [ ]:
cols_2_continue_operations_on

In [ ]:
are_correlated.describe()

## Building the SVI

### Normalizing all Features bet. 0 & 1 to make all features particicpate equally in building the SVI

In [ ]:
for col in cols_2_continue_operations_on:
    scaler = MinMaxScaler()

    scaled_values = scaler.fit_transform(X=are_correlated[col].to_numpy().reshape(-1, 1))

    are_correlated = are_correlated.with_columns(
        pl.Series(col, scaled_values.flatten(), dtype=pl.Float64)
    )

In [ ]:
are_correlated = are_correlated.with_columns(sum=pl.sum_horizontal(cols_2_continue_operations_on))

are_correlated = are_correlated.rename({"sum": "SVI_raw"})

are_correlated

In [ ]:
# Create normalized SVI column using polars expressions
are_correlated = are_correlated.with_columns([
    ((pl.col("SVI_raw") - pl.col("SVI_raw").min()) /
     (pl.col("SVI_raw").max() - pl.col("SVI_raw").min())).alias("SVI_normalized")
])



In [ ]:
ids = individuals_with_trips_df.select("ID")

out_df = pl.concat([ids, are_correlated.select(["SVI_raw", "SVI_normalized"])], how="horizontal")

out_df.write_csv("data/agents_svi_scores.csv")

In [ ]:
xx

## Critical Visualization

In [ ]:
os.makedirs('plots', exist_ok=True)
os.makedirs('plots/relationships_with_svi', exist_ok=True)
os.makedirs('plots/relationships_with_transformations', exist_ok=True)

In [ ]:
# Create a more informative scatter plot
def plot_transformation_relationship(original_data, transformed_data, feature_name):
    """
    Create a plot showing the relationship between original and transformed features

    Parameters:
    -----------
    original_data : polars.DataFrame
        DataFrame containing the original feature
    transformed_data : polars.DataFrame
        DataFrame containing the transformed feature
    feature_name : str
        Name of the feature (without 'TRANSFORMED_' prefix)
    """
    # Set style parameters
    plt.style.use('ggplot')
    fig, ax = plt.subplots(figsize=(10, 6), dpi=300)

    # Calculate means
    original_mean = original_data[feature_name].mean()
    transformed_mean = transformed_data[f"TRANSFORMED_{feature_name}"].mean()

    # Create main scatter plot
    scatter = plt.scatter(x=original_data[feature_name],
                          y=transformed_data[f"TRANSFORMED_{feature_name}"],
                          alpha=0.6,
                          label='Data Points')

    # Add mean lines
    plt.axhline(y=transformed_mean, color='red', linestyle='--',
                label=f'Mean Transformed: {transformed_mean:.2f}')
    plt.axvline(x=original_mean, color='blue', linestyle='--',
                label=f'Mean Original: {original_mean:.2f}')

    # Add colorbar scatter
    colorbar_scatter = plt.scatter(x=original_data[feature_name],
                                   y=transformed_data[f"TRANSFORMED_{feature_name}"],
                                   c=original_data[feature_name],
                                   cmap='viridis',
                                   alpha=0.6)

    # Styling
    plt.title(f'Relationship Between {feature_name} and Transformed {feature_name}')
    plt.xlabel(f'Original {feature_name}')
    plt.ylabel(f'Transformed {feature_name}')
    plt.grid(True, alpha=0.3)
    plt.legend(loc='best')

    # Add colorbar
    cbar = plt.colorbar(colorbar_scatter)
    cbar.set_label(f'Original {feature_name}')

    # Improve layout
    plt.tight_layout()

    # Save the plot
    plt.savefig(f"plots/relationships_with_transformations/transformation_{feature_name}.png", bbox_inches='tight',
                facecolor='white')
    plt.close()


# Example usage:
for feature in ["AGE", "DIPLOMA", "PRO_CAT", "NB_CAR", "NBPERS_HOUSE", "TWO_WHEELER", "BIKE", "ELECT_SCOOTER"]:
    plot_transformation_relationship(individuals_with_trips_df, are_correlated, feature)


In [ ]:
def create_styled_scatter_plot(data, feature_name, target="SVI_raw"):
    """
    Create a beautifully styled scatter plot with proper color scaling and means
    """
    # Set style parameters
    plt.style.use('ggplot')
    fig, ax = plt.subplots(figsize=(12, 8), dpi=300)

    # Calculate means
    svi_mean = data[target].drop_nans().mean()
    feature_mean = data[feature_name].mean()

    # Create main scatter plot
    original_feature = feature_name.replace('TRANSFORMED_', '')

    # Main scatter plot
    scatter = plt.scatter(x=data[feature_name],
                          y=data[target],
                          c=individuals_with_trips_df[original_feature] if 'TRANSFORMED_' in feature_name else data[
                              feature_name],
                          cmap='viridis',
                          alpha=0.6,
                          s=100,
                          label='Data Points')

    # Add mean lines
    plt.axhline(y=svi_mean, color='red', linestyle='--', label=f'Mean SVI: {svi_mean:.2f}')
    plt.axvline(x=feature_mean, color='blue', linestyle='--', label=f'Mean {feature_name}: {feature_mean:.2f}')

    # Styling
    title = f'Relationship Between {feature_name} and SVI'
    plt.title(title, pad=20, fontsize=14, fontweight='bold')
    plt.xlabel(feature_name, fontsize=12)
    plt.ylabel('Social Vulnerability Index (SVI)', fontsize=12)

    # Grid styling
    plt.grid(True, linestyle='--', alpha=0.3)

    # Add legend with both scatter points and mean lines
    plt.legend(loc='best')

    # Colorbar styling
    cbar = plt.colorbar(scatter)
    cbar.set_label(original_feature if 'TRANSFORMED_' in feature_name else feature_name,
                   fontsize=10,
                   rotation=270,
                   labelpad=15)

    # Improve layout
    plt.tight_layout()

    # Save the plot
    plt.savefig(f"plots/relationships_with_svi/{title}.png", bbox_inches='tight', facecolor='white')
    plt.close()


# Create plots for all features
for col in cols_2_continue_operations_on:
    create_styled_scatter_plot(are_correlated, col)

    # plt.show()


In [ ]:
plt.style.use('ggplot')  # Using seaborn style for better aesthetics
fig, ax = plt.subplots(figsize=(12, 8), dpi=300)

# Calculate means and other statistical measures
original_mean = individuals_with_trips_df["AGE"].mean()
svi_mean = are_correlated["SVI_raw"].mean()

# Create enhanced scatter plot
scatter = plt.scatter(x=individuals_with_trips_df["AGE"],
                      y=are_correlated["SVI_raw"],
                      alpha=0.6,
                      c=individuals_with_trips_df["AGE"],
                      cmap='viridis',
                      s=100,  # Larger point size
                      label='Individual Data Points')

# Add mean lines with improved styling
plt.axvline(x=original_mean, color='blue', linestyle='--', linewidth=2,
            label=f'Mean Age: {original_mean:.1f} years')
plt.axhline(y=svi_mean, color='red', linestyle='--', linewidth=2,
            label=f'Mean SVI: {svi_mean:.3f}')

# Enhanced styling
plt.title('Age vs Social Vulnerability Index (SVI)', fontsize=16, pad=20)
plt.xlabel('Age (years)', fontsize=12)
plt.ylabel('Social Vulnerability Index', fontsize=12)
plt.grid(True, alpha=0.3, linestyle=':')

# Improved legend
plt.legend(loc='upper right', bbox_to_anchor=(1.15, 1))

# Add colorbar with better formatting
cbar = plt.colorbar(scatter)
cbar.set_label('Age (years)', fontsize=10, rotation=270, labelpad=15)

# Add text box with statistical information
stats_text = f'Statistics:\n' \
             f'Age Range: [{individuals_with_trips_df["AGE"].min():.0f}-{individuals_with_trips_df["AGE"].max():.0f}]\n' \
             f'SVI Range: [{are_correlated["SVI_raw"].min():.2f}-{are_correlated["SVI_raw"].max():.2f}]'
plt.text(0.02, 0.98, stats_text, transform=ax.transAxes,
         bbox=dict(facecolor='white', alpha=0.8, edgecolor='gray'),
         fontsize=10, verticalalignment='top')

# Improve layout with more space for the colorbar
plt.tight_layout()

plt.savefig('plots/age_vs_svi.png', bbox_inches='tight', facecolor='white')
plt.close()

In [ ]:
are_correlated.shape

In [ ]:
plt.style.available

In [ ]:
are_correlated.describe()

#### Plotting the SVI Distribution on MAP

In [ ]:
def calculate_svi_method1(are_correlated, individuals_with_trips_df):
    """
    Calculate SVI using weighted average - most interpretable approach
    """
    # Update the dataframe with area names and weights
    are_correlated_weighted = are_correlated.with_columns([
        pl.Series("insee_code", individuals_with_trips_df["CODGEO"].cast(pl.Utf8)),
        pl.Series("AREA_NAME", individuals_with_trips_df["AREA_NAME"]),
        pl.Series("WEIGHT_INDIV", individuals_with_trips_df["WEIGHT_INDIV"].cast(pl.Float64)),
        pl.Series("svi_raw", are_correlated["SVI_raw"].cast(pl.Float64))
    ])

    # Calculate weighted SVI scores by area
    weighted_svi = are_correlated_weighted.group_by(["insee_code", "AREA_NAME"]).agg([
        pl.col("svi_raw").mul(pl.col("WEIGHT_INDIV")).sum().alias("weighted_svi_sum"),
        pl.col("WEIGHT_INDIV").sum().alias("total_weight")
    ]).with_columns([
        (pl.col("weighted_svi_sum") / pl.col("total_weight")).alias("svi_score")
    ])

    # Apply MinMaxScaler to final SVI scores for better interpretability
    min_max_scaler = MinMaxScaler()
    svi_scores_scaled = min_max_scaler.fit_transform(weighted_svi["svi_score"].to_numpy().reshape(-1, 1))

    # Add scaled scores back to dataframe
    final_svi = weighted_svi.with_columns([
        pl.Series("svi_score", svi_scores_scaled.flatten())
    ])

    return final_svi

In [ ]:

from matplotlib.colors import LinearSegmentedColormap


def create_clean_compact_svi_choropleth(svi_data, save_path=None):
    """
    Create clean, compact choropleth map with beautiful box annotations
    but without cluttered district names on the map

    Parameters:
    svi_data (DataFrame): DataFrame with 'insee_code', 'svi_score', and 'AREA_NAME' columns
    save_path (str, optional): Path to save the figure

    Returns:
    GeoDataFrame: Merged geographic maps_data with SVI scores
    """

    # Load Île-de-France communes
    geojson_url = "https://france-geojson.gregoiredavid.fr/repo/communes.geojson"
    communes_gdf = gpd.read_file(geojson_url)

    # Filter for Île-de-France departments
    idf_departments = ['75', '77', '78', '91', '92', '93', '94', '95']
    communes_gdf['dept'] = communes_gdf['code'].str[:2]
    idf_gdf = communes_gdf[communes_gdf['dept'].isin(idf_departments)].copy()

    # Process and merge SVI maps_data
    svi_grouped = svi_data.groupby(['insee_code', 'AREA_NAME'])['svi_score'].mean().reset_index()
    svi_grouped['insee_code'] = svi_grouped['insee_code'].astype(str)
    idf_gdf['code'] = idf_gdf['code'].astype(str)

    map_data = idf_gdf.merge(svi_grouped, left_on='code', right_on='insee_code', how='left')
    map_data['svi_score'] = map_data['svi_score'].fillna(0)

    # Filter to areas with maps_data for focused view
    areas_with_data = map_data[map_data['svi_score'] > 0].copy()

    # Apply ggplot style
    plt.style.use('seaborn-v0_8-whitegrid')

    # Create compact figure with high DPI
    fig, ax = plt.subplots(1, 1, figsize=(14, 10), dpi=300)

    # Set background color for ggplot feel
    fig.patch.set_facecolor('#F8F8F8')
    ax.set_facecolor('#F8F8F8')

    # Calculate percentiles for better color scaling
    svi_values = areas_with_data['svi_score']
    vmin, vmax = svi_values.quantile([0.02, 0.98]) if len(svi_values) > 0 else (0, 1)

    # Create custom colormap with better contrast
    colors = ['#2166AC', '#4393C3', '#92C5DE', '#D1E5F0', '#F7F7F7',
              '#FDBF6F', '#FF7F00', '#E31A1C', '#B10026']
    n_bins = 100
    cmap = LinearSegmentedColormap.from_list('custom', colors, N=n_bins)

    # Plot all areas in light gray first (context)
    map_data.plot(
        ax=ax,
        color='#E8E8E8',
        edgecolor='white',
        linewidth=0.2,
        alpha=0.4
    )

    # Create choropleth for areas with maps_data
    areas_with_data.plot(
        column='svi_score',
        ax=ax,
        cmap=cmap,
        edgecolor='white',
        linewidth=0.3,
        alpha=0.9,
        vmin=vmin,
        vmax=vmax,
        legend=False
    )

    # Add department boundaries with subtle styling
    dept_boundaries = map_data.dissolve(by='dept')
    dept_boundaries.boundary.plot(ax=ax, color='#2C3E50', linewidth=1.0, alpha=0.6)

    # Set view to focus on areas with maps_data (with some padding)
    if len(areas_with_data) > 0:
        bounds = areas_with_data.total_bounds
        padding = 0.03
        ax.set_xlim(bounds[0] - padding, bounds[2] + padding)
        ax.set_ylim(bounds[1] - padding, bounds[3] + padding)

    # Create custom colorbar
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=vmin, vmax=vmax))
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, shrink=0.7, aspect=25, pad=0.02)
    cbar.set_label('Social Vulnerability Index (SVI)',
                   fontsize=12, fontweight='bold', labelpad=15)
    cbar.ax.tick_params(labelsize=10)

    # Clean, compact title
    main_title = 'Social Vulnerability Index Distribution'
    subtitle = 'Île-de-France Region'

    ax.text(0.5, 0.96, main_title,
            transform=ax.transAxes, fontsize=16, fontweight='bold',
            ha='center', va='top', color='#2C3E50')
    ax.text(0.5, 0.93, subtitle,
            transform=ax.transAxes, fontsize=11, style='italic',
            ha='center', va='top', color='#5D6D7E')

    # Compact statistical annotations box
    stats_text = (
        f'📊 Dataset Overview\n'
        f'Areas Analyzed: {len(areas_with_data):,}\n'
        f'Mean SVI: {svi_values.mean():.2f}\n'
        f'Median SVI: {svi_values.median():.2f}\n'
        f'Range: {svi_values.min():.1f} - {svi_values.max():.1f}\n'
        f'Std Dev: {svi_values.std():.2f}'
    )

    ax.text(0.02, 0.95, stats_text,
            transform=ax.transAxes, fontsize=9,
            bbox=dict(boxstyle='round,pad=0.4', facecolor='white', alpha=0.95,
                      edgecolor='#BDC3C7', linewidth=1),
            verticalalignment='top',
            fontfamily='monospace')

    # Enhanced top/bottom areas legend with better formatting
    high_svi = areas_with_data.nlargest(6, 'svi_score')
    low_svi = areas_with_data.nsmallest(6, 'svi_score')

    legend_text = (
        f'Highest Vulnerability\n'
        f'{chr(10).join([f"• {row.AREA_NAME[:20]}{"..." if len(row.AREA_NAME) > 20 else ""} ({row.svi_score:.1f})" for _, row in high_svi.iterrows()])}\n\n'
        f'Lowest Vulnerability\n'
        f'{chr(10).join([f"• {row.AREA_NAME[:20]}{"..." if len(row.AREA_NAME) > 20 else ""} ({row.svi_score:.1f})" for _, row in low_svi.iterrows()])}'
    )

    ax.text(0.98, 0.95, legend_text,
            transform=ax.transAxes, fontsize=8,
            bbox=dict(boxstyle='round,pad=0.4', facecolor='#ECF0F1', alpha=0.95,
                      edgecolor='#95A5A6', linewidth=1),
            verticalalignment='top', horizontalalignment='right')

    # Compact interpretation guide
    interpretation_text = (
        'Color Scale Interpretation\n'
        'Red: High vulnerability\n'
        'Yellow: Medium vulnerability\n'
        'Blue: Low vulnerability\n'
        '\n'
        'SVI Components:\n'
        'Education levels\n'
        'Mobility resources\n'
        'Demographics\n'
        'Transportation access'
    )

    ax.text(0.02, 0.05, interpretation_text,
            transform=ax.transAxes, fontsize=8,
            bbox=dict(boxstyle='round,pad=0.4', facecolor='#E8F6F3', alpha=0.95,
                      edgecolor='#52C41A', linewidth=1),
            verticalalignment='bottom', horizontalalignment='left')

    # Compact methodology note
    methodology_text = (
        'Methodology: SVI computed using PCA-weighted socioeconomic indicators | '
        f'Coverage: {len(areas_with_data) / len(map_data) * 100:.1f}% of Île-de-France communes'
    )

    ax.text(0.5, 0.02, methodology_text,
            transform=ax.transAxes, fontsize=7, style='italic',
            ha='center', va='bottom', color='#7F8C8D',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.9))

    # Remove axes for cleaner look
    ax.set_axis_off()

    # Add subtle grid lines for ggplot feel
    ax.grid(True, alpha=0.1, linestyle='-', linewidth=0.3, color='#BDC3C7')

    # Add compact north arrow
    ax.annotate('N', xy=(0.96, 0.88), xytext=(0.96, 0.85),
                xycoords='axes fraction', textcoords='axes fraction',
                fontsize=12, fontweight='bold', ha='center',
                arrowprops=dict(arrowstyle='->', color='#2C3E50', lw=1.5))

    # Tight layout with minimal padding
    plt.tight_layout(pad=0.5)

    # Save with high quality if path provided
    if save_path:
        plt.savefig(save_path, dpi=400, bbox_inches='tight',
                    facecolor='#F8F8F8', edgecolor='none',
                    format='png')
        print(f"Clean compact map saved to {save_path}")

    plt.show()

    # Clean, compact statistics output
    print("\n" + "=" * 60)
    print("📊 SVI DISTRIBUTION ANALYSIS")
    print("=" * 60)
    print(f"🗺️  Areas Analyzed: {len(areas_with_data)} communes")
    print(f"📈 Coverage: {len(areas_with_data) / len(map_data) * 100:.1f}% of Île-de-France")
    print(f"📊 Mean SVI: {svi_values.mean():.2f} ± {svi_values.std():.2f}")
    print(f"📊 Range: {svi_values.min():.1f} - {svi_values.max():.1f}")

    print(f"\n🔴 TOP 5 MOST VULNERABLE AREAS:")
    print("-" * 45)
    for i, (_, row) in enumerate(high_svi.head(5).iterrows(), 1):
        print(f"{i}. {row['AREA_NAME']:25} | {row['svi_score']:5.2f}")

    print(f"\n🔵 TOP 5 LEAST VULNERABLE AREAS:")
    print("-" * 45)
    for i, (_, row) in enumerate(low_svi.head(5).iterrows(), 1):
        print(f"{i}. {row['AREA_NAME']:25} | {row['svi_score']:5.2f}")

    print("=" * 60)

    return areas_with_data


# =============================================================================
# CLEAN COMPACT EXECUTION
# =============================================================================
result_method1 = calculate_svi_method1(are_correlated, individuals_with_trips_df)
svi_data = result_method1.select(['insee_code', 'svi_score', 'AREA_NAME']).to_pandas()

# Create the clean compact map
print("🗺️  Creating Clean Compact SVI Choropleth Map...")
print("⚙️  Features: Beautiful box annotations, clean design, focused view")
print("🎯 Design: Compact layout without cluttered district names")

result = create_clean_compact_svi_choropleth(svi_data, save_path='clean_compact_svi_map.png')

print("\n✅ Clean compact visualization complete!")
print("📊 Beautiful box annotations preserved with cleaner map design")

## Intuitions & Brainstorming

In [ ]:
#1st intuition 1/1+log(age) for ages lower than mean and log(age)-1 for ages above
"""
mean_age = 43
Age 42: 1 / (1 + log(42)) = 1 / (1 + 3.74) = 0.211
Age 44: log(44) - 1 = 3.78 - 1 = 2.78
"""

#2nd idea log(1 + |AGE - μ|) produces u-shaped snooth curve

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Set the style
plt.style.use('ggplot')

# Create maps_data points
i_s = np.linspace(0, 100, 1000)  # More points for smoother curve
product = 1 / (1 + np.log1p(1 + i_s))  # np.log1p is more numerically stable than log(1+x)

# Create the plot
plt.figure(figsize=(12, 7))

# Main plot with gradient color
scatter = plt.scatter(i_s, product,
                      c=i_s,  # Color points by i value
                      cmap='viridis',  # Use viridis colormap
                      alpha=0.6,
                      s=30)  # Point size

# Add colorbar
plt.colorbar(scatter, label='x value')

# Customize the plot
plt.title('Relationship Between x and 1/(1+log(x))', fontsize=14, pad=20)
plt.xlabel('x', fontsize=12)
plt.ylabel('1/(1+log(x))', fontsize=12)

# Add grid with custom style
plt.grid(True, linestyle='--', alpha=0.3)

# Add annotations for key points
plt.annotate(f'As x → ∞, y → 0',
             xy=(80, 0.2),
             xytext=(60, 0.3),
             arrowprops=dict(facecolor='black', shrink=0.05),
             bbox=dict(boxstyle='round,pad=0.5', fc='yellow', alpha=0.3))

# Add mathematical formula
plt.text(0.02, 0.98, r'$f(x) = \frac{1}{1 + \log(1+x)}$',
         transform=plt.gca().transAxes,
         fontsize=12,
         bbox=dict(facecolor='white', edgecolor='none', alpha=0.7))

# Improve layout
plt.tight_layout()

# Show plot
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Set the style with a modern look
plt.style.use('ggplot')

# Create maps_data points with careful range
x = np.linspace(0, 100, 1000)
y = np.log1p(x)  # Using log1p for numerical stability

# Create figure and axis objects with specific size
fig, ax = plt.subplots(figsize=(12, 8))

# Create gradient color mapping
norm = plt.Normalize(x.min(), x.max())
colors = plt.cm.viridis(norm(x))

# Main scatter plot with gradient
scatter = ax.scatter(x, y, c=x, cmap='viridis', alpha=0.6, s=40)

# Customize grid
ax.grid(True, linestyle='--', alpha=0.3)

# Add colorbar with custom styling
cbar = plt.colorbar(scatter)
cbar.set_label('x value', fontsize=12, labelpad=15)
cbar.ax.tick_params(labelsize=10)

# Set labels and title with LaTeX formatting
ax.set_title('Relationship Between x and log(x+1)', fontsize=16, pad=20)
ax.set_xlabel('x', fontsize=14, labelpad=10)
ax.set_ylabel('log(x+1)', fontsize=14, labelpad=10)

# Add tick parameters
ax.tick_params(axis='both', which='major', labelsize=10)

# Add key points annotation
ax.annotate('Logarithmic growth:\nIncreases quickly for small x,\n'
            'then gradually slows down',
            xy=(20, 3),
            xytext=(40, 2),
            fontsize=10,
            bbox=dict(facecolor='white', edgecolor='gray', alpha=0.8),
            arrowprops=dict(arrowstyle='->',
                            connectionstyle='arc3,rad=-.2',
                            color='gray'))

# Add mathematical formula with LaTeX
ax.text(0.02, 0.95, r'$f(x) = log(x+1)$',
        transform=ax.transAxes,
        fontsize=14,
        bbox=dict(facecolor='white',
                  edgecolor='gray',
                  alpha=0.8,
                  pad=10))

# Add horizontal asymptote visualization
ax.axhline(y=np.log1p(100), color='red', linestyle='--', alpha=0.3)
ax.text(80, np.log1p(100) + 0.1, 'Asymptotic behavior',
        fontsize=10, color='red')

# Improve layout
plt.tight_layout()

# Show plot
plt.show()

In [ ]:
"""Indicator Group	Variable(s) from Dataset	What It Measures (Proxy for...)
Physical Capacity	PMR, AGE	Direct physical barriers to mobility
Household Resources	NB_CAR, DRIVING_LICENCE, TWO_WHEELER etc.	Access to private, independent transport options
Household Constraints	NBPERS_HOUSE, NB_10, etc.	Presence of dependents complicating evacuation
Socio-Economic Status	DIPLOMA, PRO_CAT	Income, job flexibility, information access
System Dependency	NAVIGO_SUB, etc.	Reliance on public systems that might fail
"""

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from pynsee.geodata import get_geodata, get_geodata_list
import os

os.environ['sirene_key'] = '8443bcd3-ebf5-4a54-83bc-d3ebf51a546f'

# Step 1: Explore available geodata and find commune identifiers
print("Available geodata types:")
geodata_list = get_geodata_list()

# Find commune-related identifiers
commune_identifiers = []
for _, row in geodata_list.iterrows():
    if 'commune' in row['Identifier'].lower():
        commune_identifiers.append(row['Identifier'])

print(f"\nFound {len(commune_identifiers)} commune identifiers:")
for identifier in commune_identifiers:
    print(f"  - {identifier}")


# Helper function to find the best column name
def find_column_by_pattern(df, patterns):
    """Find a column name that matches any of the given patterns"""
    for col in df.columns:
        for pattern in patterns:
            if pattern.lower() in col.lower():
                return col
    return None


# Step 2: Load your SVI maps_data (replace with your actual maps_data loading)
# Assuming your maps_data has columns: 'insee_code' and 'svi_score'
# svi_data = pd.read_csv('your_svi_data.csv')  # Replace with your actual file

# For demonstration, let's create sample maps_data
# In practice, replace this with your actual maps_data loading
sample_insee_codes = ['75001', '75002', '75003', '92001', '92002', '93001', '94001', '95001']
sample_svi_scores = [150, 200, 180, 220, 190, 250, 210, 170]
svi_data = pd.DataFrame({
    'insee_code': sample_insee_codes,
    'svi_score': sample_svi_scores
})

# Step 3: Group by INSEE codes and sum SVI scores
# If you have multiple rows per INSEE code, this will aggregate them
svi_grouped = svi_data.groupby('insee_code')['svi_score'].sum().reset_index()
print(f"Grouped SVI maps_data shape: {svi_grouped.shape}")
print(svi_grouped.head())

# Step 4: Get geographic maps_data for Île-de-France
# From the geodata list, we need to use the correct identifier
print("\nLooking for commune maps_data in geodata list...")

# Get all communes in France first (we'll filter for Île-de-France later)
# Using the identifier from the geodata list
try:
    # Try the most recent commune maps_data
    idf_geodata = get_geodata('ADMINEXPRESS-COG.LATEST:commune')
    print(f"\nGeodata shape: {idf_geodata.shape}")
    print(f"Columns: {idf_geodata.columns.tolist()}")

    # Display sample of geodata
    print("\nSample of geodata:")
    print(idf_geodata.head())

    # Filter for Île-de-France (region code 11)
    # The region code is usually in a column like 'reg' or 'region'
    region_col = None
    for col in idf_geodata.columns:
        if 'reg' in col.lower():
            region_col = col
            break

    if region_col:
        print(f"\nUsing region column: {region_col}")
        print(f"Unique region values: {sorted(idf_geodata[region_col].unique())}")

        # Filter for Île-de-France (region code 11)
        idf_geodata = idf_geodata[idf_geodata[region_col] == '11'].copy()
        print(f"Filtered maps_data shape for Île-de-France: {idf_geodata.shape}")
    else:
        print("Could not find region column, using all communes")

except Exception as e:
    print(f"Error loading geodata: {e}")
    # Try alternative identifier
    try:
        print("Trying alternative identifier...")
        idf_geodata = get_geodata('ADMINEXPRESS-COG.2017:commune')
        print(f"Alternative geodata shape: {idf_geodata.shape}")
        print(f"Columns: {idf_geodata.columns.tolist()}")

        # Filter for Île-de-France if region column exists
        region_col = None
        for col in idf_geodata.columns:
            if 'reg' in col.lower():
                region_col = col
                break

        if region_col:
            idf_geodata = idf_geodata[idf_geodata[region_col] == '11'].copy()
            print(f"Filtered maps_data shape for Île-de-France: {idf_geodata.shape}")

    except Exception as e2:
        print(f"Alternative approach also failed: {e2}")
        print("Please check available identifiers in the geodata list")

# Step 5: Merge SVI maps_data with geographic maps_data
# Find the INSEE code column using multiple possible patterns
insee_patterns = ['insee', 'com', 'code_insee', 'depcom', 'insee_com']
insee_col = find_column_by_pattern(idf_geodata, insee_patterns)

if insee_col:
    print(f"\nUsing INSEE column: {insee_col}")
    print(f"Sample INSEE codes: {idf_geodata[insee_col].head().tolist()}")

    # Ensure INSEE codes are strings and properly formatted
    idf_geodata[insee_col] = idf_geodata[insee_col].astype(str)
    svi_grouped['insee_code'] = svi_grouped['insee_code'].astype(str)

    # Merge the maps_data
    map_data = idf_geodata.merge(
        svi_grouped,
        left_on=insee_col,
        right_on='insee_code',
        how='left'
    )

    # Fill NaN values with 0 for areas without SVI maps_data
    map_data['svi_score'] = map_data['svi_score'].fillna(0)

    print(f"Merged maps_data shape: {map_data.shape}")
    print(f"Areas with SVI maps_data: {(map_data['svi_score'] > 0).sum()}")

    # Show which INSEE codes from your maps_data were matched
    matched_codes = map_data[map_data['svi_score'] > 0]['insee_code'].unique()
    unmatched_codes = set(svi_grouped['insee_code']) - set(matched_codes)

    print(f"Matched INSEE codes: {sorted(matched_codes)}")
    if unmatched_codes:
        print(f"Unmatched INSEE codes: {sorted(unmatched_codes)}")

else:
    print("Could not find INSEE code column in geodata")
    print("Available columns:", idf_geodata.columns.tolist())

    # If we can't find the INSEE column, show some sample maps_data to help debug
    print("\nSample of geodata to help identify the right column:")
    print(idf_geodata.head())

    # Exit early if we can't find the column
    print("Please check the column names and update the code accordingly.")
    exit()

# Step 6: Create the choropleth map
if 'map_data' in locals() and not map_data.empty:
    fig, ax = plt.subplots(1, 1, figsize=(12, 10))

    # Create the choropleth map
    map_data.plot(
        column='svi_score',
        ax=ax,
        legend=True,
        cmap='YlOrRd',  # Color map (yellow to red)
        edgecolor='black',
        linewidth=0.1,
        legend_kwds={
            'label': 'SVI Score',
            'orientation': 'vertical',
            'shrink': 0.6
        }
    )

    # Customize the map
    ax.set_title('SVI Scores by District in Île-de-France', fontsize=16, fontweight='bold')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')

    # Remove axis ticks for cleaner look
    ax.tick_params(axis='both', which='major', labelsize=8)

    # Add grid
    ax.grid(True, alpha=0.3)

    # Tight layout
    plt.tight_layout()

    # Show the map
    plt.show()

    # Step 7: Print summary statistics
    print(f"\nSVI Score Statistics:")
    print(f"Mean: {map_data['svi_score'].mean():.2f}")
    print(f"Median: {map_data['svi_score'].median():.2f}")
    print(f"Min: {map_data['svi_score'].min():.2f}")
    print(f"Max: {map_data['svi_score'].max():.2f}")
    print(f"Standard deviation: {map_data['svi_score'].std():.2f}")

    # Optional: Save the map
    # plt.savefig('idf_svi_map.png', dpi=300, bbox_inches='tight')

else:
    print("Cannot create map - maps_data merging failed or no maps_data available.")

In [ ]:


# Optional: Interactive map with Folium (uncomment to use)
"""
import folium

def create_interactive_map(map_data):
    # Create base map centered on Paris
    m = folium.Map(location=[48.8566, 2.3522], zoom_start=10)

    # Add choropleth layer
    folium.Choropleth(
        geo_data=map_data,
        maps_data=map_data,
        columns=['code', 'svi_score'],
        key_on='feature.properties.code',
        fill_color='YlOrRd',
        fill_opacity=0.7,
        line_opacity=0.2,
        legend_name='SVI Score'
    ).add_to(m)

    # Add tooltips
    folium.features.GeoJsonTooltip(
        fields=['nom', 'code', 'svi_score'],
        aliases=['Commune:', 'INSEE Code:', 'SVI Score:'],
        localize=True
    ).add_to(m)

    return m

# Create interactive map
# interactive_map = create_interactive_map(result)
# interactive_map.save('idf_svi_interactive_map.html')
"""